# XGBoost Training

Baseline training and fine-tuning workflow for the XGBoost model used in the `distraction_prediction` module.

## Imports

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)


## Paths

In [2]:
def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / 'distraction_prediction' / 'data' / 'processed' / 'windows').exists():
            return candidate
    raise FileNotFoundError('Could not locate project root containing distraction_prediction/data/processed/windows')


PROJECT_ROOT = find_project_root()
WINDOWS_DIR = PROJECT_ROOT / 'distraction_prediction' / 'data' / 'processed' / 'windows'
SAVE_DIR = PROJECT_ROOT / 'distraction_prediction' / 'models' / 'saved_models' / 'baselines'
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Windows dir  :', WINDOWS_DIR)
print('Save dir     :', SAVE_DIR)


Project root : e:\SDPPS
Windows dir  : e:\SDPPS\distraction_prediction\data\processed\windows
Save dir     : e:\SDPPS\distraction_prediction\models\saved_models\baselines


## Helper Functions

In [3]:
def load_window_splits():
    return {
        'X_train': np.load(WINDOWS_DIR / 'X_train.npy'),
        'y_train': np.load(WINDOWS_DIR / 'y_train.npy'),
        'X_val': np.load(WINDOWS_DIR / 'X_val.npy'),
        'y_val': np.load(WINDOWS_DIR / 'y_val.npy'),
        'X_test': np.load(WINDOWS_DIR / 'X_test.npy'),
        'y_test': np.load(WINDOWS_DIR / 'y_test.npy'),
    }


def flatten_windows(X_3d):
    mean_features = np.mean(X_3d, axis=1)
    last_features = X_3d[:, -1, :]
    return np.hstack([mean_features, last_features]).astype(np.float32)


def prepare_flattened_splits():
    data = load_window_splits()
    return {
        'X_train': flatten_windows(data['X_train']),
        'y_train': data['y_train'],
        'X_val': flatten_windows(data['X_val']),
        'y_val': data['y_val'],
        'X_test': flatten_windows(data['X_test']),
        'y_test': data['y_test'],
    }


def compute_scale_pos_weight(y):
    positives = int((y == 1).sum())
    negatives = int((y == 0).sum())
    return negatives / positives if positives > 0 else 1.0


def evaluate_binary_classifier(model, X, y):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]
    matrix = confusion_matrix(y, predictions)
    return {
        'accuracy': accuracy_score(y, predictions),
        'f1': f1_score(y, predictions, zero_division=0),
        'auc': roc_auc_score(y, probabilities) if len(np.unique(y)) > 1 else 0.0,
        'confusion_matrix': matrix.tolist(),
        'classification_report': classification_report(
            y,
            predictions,
            target_names=['Focused', 'Distracted'],
            zero_division=0,
        ),
    }


def print_dataset_summary(data):
    print('Loaded flattened datasets')
    print(f"  Train: {data['X_train'].shape} | Positives: {int(data['y_train'].sum())}")
    print(f"  Val  : {data['X_val'].shape} | Positives: {int(data['y_val'].sum())}")
    print(f"  Test : {data['X_test'].shape} | Positives: {int(data['y_test'].sum())}")


def print_metrics(title, metrics):
    cm = metrics['confusion_matrix']
    print('\n' + '=' * 44)
    print(title)
    print('=' * 44)
    print(f"Accuracy : {metrics['accuracy']:.4f}")
    print(f"F1 Score : {metrics['f1']:.4f}")
    print(f"ROC-AUC  : {metrics['auc']:.4f}")
    print('Confusion Matrix:')
    print(f"  TN={cm[0][0]}  FP={cm[0][1]}")
    print(f"  FN={cm[1][0]}  TP={cm[1][1]}")
    print('=' * 44)
    print(metrics['classification_report'])


def save_model_and_results(model, model_name, results_name, results):
    joblib.dump(model, SAVE_DIR / model_name)
    serializable_results = {}
    for key, value in results.items():
        if isinstance(value, (np.floating, float)):
            serializable_results[key] = round(float(value), 4)
        else:
            serializable_results[key] = value
    with open(SAVE_DIR / results_name, 'w', encoding='utf-8') as file:
        json.dump(serializable_results, file, indent=2)


## Load Prepared Window Data

In [4]:
data = prepare_flattened_splits()
print_dataset_summary(data)

scale_pos_weight = compute_scale_pos_weight(data['y_train'])
print('scale_pos_weight =', round(scale_pos_weight, 2))


Loaded flattened datasets
  Train: (51371, 48) | Positives: 29477
  Val  : (6421, 48) | Positives: 3685
  Test : (6422, 48) | Positives: 3685
scale_pos_weight = 0.74


## Baseline XGBoost

In [5]:
BASELINE_CONFIG = {
    'n_estimators': 300,
    'max_depth': 6,
    'learning_rate': 0.05,
    'early_stopping_rounds': 20,
    'eval_metric': 'logloss',
    'random_state': 42,
}

baseline_model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, **BASELINE_CONFIG)
baseline_model.fit(
    data['X_train'],
    data['y_train'],
    eval_set=[(data['X_val'], data['y_val'])],
    verbose=10,
)

baseline_metrics = evaluate_binary_classifier(baseline_model, data['X_test'], data['y_test'])
print_metrics('Baseline XGBoost Results', baseline_metrics)


[0]	validation_0-logloss:0.67033
[10]	validation_0-logloss:0.52859
[20]	validation_0-logloss:0.46402
[30]	validation_0-logloss:0.42998
[40]	validation_0-logloss:0.41147
[50]	validation_0-logloss:0.39951
[60]	validation_0-logloss:0.39156
[70]	validation_0-logloss:0.38546
[80]	validation_0-logloss:0.38123
[90]	validation_0-logloss:0.37730
[100]	validation_0-logloss:0.37264
[110]	validation_0-logloss:0.36918
[120]	validation_0-logloss:0.36653
[130]	validation_0-logloss:0.36393
[140]	validation_0-logloss:0.36151
[150]	validation_0-logloss:0.35944
[160]	validation_0-logloss:0.35691
[170]	validation_0-logloss:0.35519
[180]	validation_0-logloss:0.35300
[190]	validation_0-logloss:0.35050
[200]	validation_0-logloss:0.34807
[210]	validation_0-logloss:0.34599
[220]	validation_0-logloss:0.34316
[230]	validation_0-logloss:0.34111
[240]	validation_0-logloss:0.33949
[250]	validation_0-logloss:0.33741
[260]	validation_0-logloss:0.33557
[270]	validation_0-logloss:0.33372
[280]	validation_0-logloss:0.33

In [6]:
save_model_and_results(
    model=baseline_model,
    model_name='xgboost.joblib',
    results_name='xgboost_results.json',
    results=baseline_metrics,
)


## Fine-Tuned XGBoost

In [7]:
SEARCH_CONFIGS = [
    {'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 1.0, 'colsample_bytree': 1.0},
    {'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8},
    {'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9},
    {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 1.0, 'colsample_bytree': 1.0},
    {'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7},
]

best_f1 = -1.0
best_model = None
best_config = None

for config in SEARCH_CONFIGS:
    model = xgb.XGBClassifier(
        n_estimators=config['n_estimators'],
        max_depth=config['max_depth'],
        learning_rate=config['learning_rate'],
        subsample=config['subsample'],
        colsample_bytree=config['colsample_bytree'],
        scale_pos_weight=scale_pos_weight,
        early_stopping_rounds=20,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
    )
    model.fit(
        data['X_train'],
        data['y_train'],
        eval_set=[(data['X_val'], data['y_val'])],
        verbose=False,
    )
    validation_metrics = evaluate_binary_classifier(model, data['X_val'], data['y_val'])
    val_f1 = validation_metrics['f1']
    print(config, 'Validation F1 =', round(val_f1, 4))

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_model = model
        best_config = config

print('Best configuration:', best_config)


{'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 1.0, 'colsample_bytree': 1.0} Validation F1 = 0.8571
{'n_estimators': 500, 'max_depth': 8, 'learning_rate': 0.03, 'subsample': 0.8, 'colsample_bytree': 0.8} Validation F1 = 0.8964
{'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9} Validation F1 = 0.9302
{'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.1, 'subsample': 1.0, 'colsample_bytree': 1.0} Validation F1 = 0.8494
{'n_estimators': 500, 'max_depth': 6, 'learning_rate': 0.01, 'subsample': 0.8, 'colsample_bytree': 0.7} Validation F1 = 0.8324
Best configuration: {'n_estimators': 400, 'max_depth': 10, 'learning_rate': 0.05, 'subsample': 0.9, 'colsample_bytree': 0.9}


In [8]:
final_metrics = evaluate_binary_classifier(best_model, data['X_test'], data['y_test'])
print_metrics('Fine-Tuned XGBoost Results', final_metrics)



Fine-Tuned XGBoost Results
Accuracy : 0.9195
F1 Score : 0.9283
ROC-AUC  : 0.9777
Confusion Matrix:
  TN=2559  FP=178
  FN=339  TP=3346
              precision    recall  f1-score   support

     Focused       0.88      0.93      0.91      2737
  Distracted       0.95      0.91      0.93      3685

    accuracy                           0.92      6422
   macro avg       0.92      0.92      0.92      6422
weighted avg       0.92      0.92      0.92      6422



In [9]:
save_model_and_results(
    model=best_model,
    model_name='xgboost_finetuned.joblib',
    results_name='xgboost_finetuned_results.json',
    results={
        **final_metrics,
        'best_config': best_config,
    },
)
